<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 05. Evaluación de Clustering: ¿Qué Tan Bueno Es Mi Agrupamiento?
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 10
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/Para%20Dummies/05_Evaluacion_Seleccion_K_y_Benchmark_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del módulo 05 de Clustering — el cuaderno que **cierra** el módulo. A lo largo de los cuadernos 02, 03 y 04 usamos K, `eps` y métodos de enlace más o menos "porque sí" — aquí aprenderemos una forma sistemática de decidir si un agrupamiento es bueno, y de elegir el número de grupos sin adivinar a ojo.

Al terminar podrás explicar, con tus propias palabras:
1. Por qué evaluar un clustering es más difícil que evaluar un modelo de clasificación (no hay "respuesta correcta" para comparar).
2. Qué mide el **coeficiente de silueta**, con una analogía simple.
3. Cómo funciona el **método del codo** para elegir K.
4. Cómo se comportan K-Means, clustering jerárquico y DBSCAN lado a lado en distintas formas de datos.


---
## 1. El problema de calificar sin respuestas correctas 🧭

En un examen de selección múltiple, calificar es fácil: comparas la respuesta del estudiante contra la respuesta correcta. En clustering **no existe** esa hoja de respuestas — nadie nos dice de antemano "estos 200 clientes en realidad forman 5 grupos". Por eso necesitamos formas de evaluar un agrupamiento **mirando solo la posición de los propios puntos**, sin ninguna etiqueta externa.

La pregunta que responderemos con esas métricas es siempre la misma, aunque cambien los números: **¿quedaron los puntos de un mismo grupo bien "apretaditos" entre sí, y bien separados de los otros grupos?**


---
## Configuración del entorno de trabajo 🛠️

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_moons, make_circles, make_blobs

df_mall = pd.read_csv('../data/mall_customers.csv')
X = df_mall[['Annual_Income_k', 'Spending_Score']].values
X_escalado = StandardScaler().fit_transform(X)

print("Dataset cargado:", df_mall.shape)

### 🤔 ¿Qué acaba de pasar?

- Volvemos a cargar `mall_customers.csv` y, esta vez, sí lo estandarizamos con `StandardScaler` — las métricas que veremos a continuación, igual que K-Means, se basan en distancias, así que conviene trabajar en la misma escala para las dos columnas.


---
## 2. El coeficiente de silueta: ¿encajas bien en tu grupo? 🧩

Imagina que estás en una fiesta dividida en varias mesas (grupos). Para cada persona, la **silueta** compara dos cosas:

* **a)** qué tan cerca está, en promedio, de la **gente de su propia mesa**.
* **b)** qué tan cerca está, en promedio, de la gente de la **mesa vecina más cercana** (la otra mesa más parecida a la suya).

Si estás mucho más cerca de tu propia mesa que de cualquier otra, tu silueta es alta (cercana a **1**): encajas bien donde estás. Si estás casi igual de cerca de tu mesa que de la de al lado, tu silueta es cercana a **0**: estás justo en la frontera entre dos grupos. Si en realidad estás más cerca de la mesa vecina que de la tuya, tu silueta es **negativa**: probablemente te sentaron en la mesa equivocada.

El **coeficiente de silueta** del agrupamiento completo es simplemente el promedio de esa "comodidad" de todas las personas — va de -1 (fatal) a 1 (excelente).


In [ ]:
modelo_kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
grupo_kmeans = modelo_kmeans.fit_predict(X_escalado)

silueta_k5 = silhouette_score(X_escalado, grupo_kmeans)
print(f"Coeficiente de silueta (K-Means, K=5): {silueta_k5:.3f}")

### 🤔 ¿Qué acaba de pasar?

- `silhouette_score` calculó, para cada uno de los 200 clientes, "qué tan cómodo" está en su grupo (comparado con el grupo vecino más cercano), y promedió el resultado.
- Un valor alrededor de 0.5-0.6 (como sueles obtener con K=5 en este dataset) indica una estructura de grupos razonablemente clara — no perfecta, pero bastante bien definida.
- ⚠️ La silueta tiene un sesgo importante: premia grupos redondeados y bien separados — el mismo tipo de forma que K-Means ya busca por construcción. Con formas como las medias lunas del cuaderno 04, la silueta puede no reflejar del todo bien qué tan "correcto" es un agrupamiento a simple vista.


---
## 3. Eligiendo K sin adivinar: el método del codo 📐

En los cuadernos anteriores usamos K=5 porque, con solo dos columnas, se ven "a ojo" 5 nubes de puntos — pero en datasets reales, con muchas más columnas, no podemos simplemente mirar un gráfico. El **método del codo** ofrece una forma sistemática:

1. Probamos K-Means con varios valores de K (por ejemplo, de 2 a 10).
2. Para cada K, guardamos la **inercia** (qué tan "apretados" quedan los grupos — mientras más bajo, mejor, pero siempre baja al aumentar K).
3. Graficamos inercia contra K, y buscamos el **codo**: el punto donde agregar un grupo más deja de mejorar mucho las cosas.

De paso, hacemos lo mismo con la silueta — con la diferencia de que ahí buscamos el **pico más alto**, no un codo, porque en la silueta "más alto es mejor".


In [ ]:
valores_k = range(2, 11)
inercias = []
siluetas = []

for k in valores_k:
    modelo = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_escalado)
    inercias.append(modelo.inertia_)
    siluetas.append(silhouette_score(X_escalado, modelo.labels_))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(list(valores_k), inercias, marker='o', color='#b45309')
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inercia")
axes[0].set_title("Método del codo (buscamos el codo)", fontweight='bold')

axes[1].plot(list(valores_k), siluetas, marker='o', color='#1e3a8a')
axes[1].set_xlabel("K")
axes[1].set_ylabel("Coeficiente de silueta")
axes[1].set_title("Silueta por K (buscamos el pico)", fontweight='bold')

plt.tight_layout()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- En el gráfico de la izquierda, la inercia baja rápido al principio (agregar el segundo, tercer o cuarto grupo ayuda mucho) y luego la curva se "aplana" — ese punto donde deja de bajar tan rápido es el codo, y suele coincidir alrededor de K=5 para este dataset, justo el valor que veníamos usando por conveniencia visual desde el cuaderno 02.
- En el gráfico de la derecha buscamos el pico más alto de silueta, no un codo — a veces coincide exactamente con el codo de inercia, y a veces sugiere un K distinto (por ejemplo, uno más pequeño), porque cada métrica pondera "compacidad" y "separación" de forma ligeramente distinta.
- **Lección importante:** ningún método es infalible por sí solo. Lo más sano es combinar el método del codo, la silueta, y el conocimiento del negocio (por ejemplo, cuántos segmentos de clientes tiene sentido manejar en la práctica) antes de decidir un K final.


---
## 4. Benchmark: tres algoritmos, tres formas de datos 🧪

Cerramos con la pregunta más importante del módulo: **¿cuál algoritmo conviene usar?** La respuesta corta es "depende de la forma real de tus datos". Comparemos K-Means, clustering jerárquico (`ward`) y DBSCAN sobre tres datasets sintéticos con formas muy distintas: blobs redondos (el caso fácil), medias lunas, y círculos concéntricos.


In [ ]:
n_muestras = 400
datasets_sinteticos = {
    'Blobs (redondos)': make_blobs(n_samples=n_muestras, centers=3, random_state=8)[0],
    'Medias lunas': make_moons(n_samples=n_muestras, noise=0.06, random_state=8)[0],
    'Círculos concéntricos': make_circles(n_samples=n_muestras, factor=0.5, noise=0.05, random_state=8)[0],
}

fig, axes = plt.subplots(3, 3, figsize=(11, 10))

for fila, (nombre_ds, X_ds) in enumerate(datasets_sinteticos.items()):
    X_ds = StandardScaler().fit_transform(X_ds)
    k = 3 if nombre_ds == 'Blobs (redondos)' else 2

    algoritmos = [
        ('K-Means', KMeans(n_clusters=k, n_init=10, random_state=0)),
        ('Jerárquico (ward)', AgglomerativeClustering(n_clusters=k, linkage='ward')),
        ('DBSCAN', DBSCAN(eps=0.25, min_samples=5)),
    ]

    for col, (nombre_algo, modelo) in enumerate(algoritmos):
        ax = axes[fila, col]
        etiquetas = modelo.fit_predict(X_ds)
        ax.scatter(X_ds[:, 0], X_ds[:, 1], c=etiquetas, cmap='viridis', s=15, alpha=0.85)
        ax.set_xticks([])
        ax.set_yticks([])
        if fila == 0:
            ax.set_title(nombre_algo, fontweight='bold')
        if col == 0:
            ax.set_ylabel(nombre_ds, fontsize=9)

plt.suptitle("Benchmark: K-Means vs. Jerárquico vs. DBSCAN en tres formas de datos", y=1.01)
plt.tight_layout()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- En los **blobs redondos** (fila de arriba), los tres algoritmos coinciden en encontrar prácticamente el mismo agrupamiento — cuando la estructura real ya es convexa y está bien separada, no importa mucho cuál algoritmo elijas.
- En **medias lunas** y **círculos concéntricos** (las dos filas de abajo), K-Means y el clustering jerárquico con `ward` cortan las formas de manera poco natural, porque ambos tienden a preferir grupos redondeados. **DBSCAN**, en cambio, sigue la forma real de las zonas densas y las separa correctamente.
- La lección de fondo de todo el módulo: **no existe un algoritmo "mejor" en general** — K-Means y el clustering jerárquico son excelentes cuando esperas grupos razonablemente compactos (como los segmentos de clientes del centro comercial); DBSCAN brilla cuando sospechas formas irregulares o la presencia de valores atípicos genuinos. La elección correcta depende siempre de cómo se ven (o se sospecha que se ven) tus datos, no de una preferencia por defecto.


---
## 5. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Evaluar sin etiquetas | En clustering no hay "respuesta correcta" — evaluamos mirando qué tan apretados y separados quedan los grupos. |
| Coeficiente de silueta | Compara qué tan cerca estás de tu propio grupo frente al grupo vecino más cercano; va de -1 a 1. |
| Método del codo | Grafica inercia (o silueta) contra K y busca el punto donde agregar más grupos deja de ayudar mucho. |
| K-Means | Rápido y sencillo; funciona bien con grupos redondeados y de tamaño similar. |
| Clustering jerárquico | No exige fijar K de antemano; expone estructura a distintos niveles mediante el dendrograma. |
| DBSCAN | Detecta formas arbitrarias y valores atípicos genuinos, sin forzar a todo punto a pertenecer a un grupo. |
| Lección del módulo | Ningún algoritmo ni ninguna métrica es universalmente "mejor" — la elección depende de la forma real de tus datos. |


---
## 🎓 ¡Felicitaciones! Llegaste al final de la serie "Para Dummies" 🎉

Con este cuaderno cerramos no solo el **Módulo 10: Clustering**, sino **toda la serie "Para Dummies"** que acompañó el curso completo de *Programación para Ciencia de Datos*, desde las primeras nociones de Python hasta este último algoritmo de aprendizaje no supervisado.

Repasa el camino que recorriste, siempre "traducido" a analogías sencillas:

* Aprendiste a pensar en arrays y tablas de datos (NumPy y Pandas).
* Exploraste y limpiaste datos con EDA y Data Preparation.
* Construiste características nuevas con Feature Engineering.
* Entrenaste y evaluaste modelos de Regresión, Clasificación y Árboles de Decisión.
* Y, en este último módulo, aprendiste a encontrar estructura en datos que **no tienen ninguna etiqueta** — K-Means, clustering jerárquico, DBSCAN, y cómo evaluar cuál conviene usar en cada caso.

Si llegaste hasta aquí leyendo cada cuaderno "para no ingenieros", ya tienes las bases conceptuales para volver a los cuadernos principales del curso con mucha más confianza — las fórmulas y el código ya no deberían sonar a idioma desconocido, porque ahora entiendes **la idea** que hay detrás de cada una.

¡Gracias por acompañar esta serie hasta el final, y mucho éxito en lo que sigue de tu camino en Ciencia de Datos! 🚀


---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
